#  XBRL Parsing — Generation 2 (SEBI Format)

## Purpose
Parses **Gen2 XBRL files** (SEBI format, post Jan 2025) filed by Nifty 50 companies
and converts them into **lean structured JSON** ready for chunking pipeline.

## Pipeline Position
Raw XBRL (data/raw/) →  This Notebook →  Lean JSON (data/parsed/gen2/) →  02_chunking.ipynb

## Input → Output
- **Input:** Raw `.xml` files (Gen2 SEBI format)
- **Output:** Lean `.json` files — one per XML with correct rounding, quarter detection and clean facts

## 1. Imports & Libraries

In [220]:
# Standard library
import json
import re
from pathlib import Path
from datetime import datetime
from typing import Any, Dict, List, Optional, Tuple
from collections import defaultdict

# XML parsing - lxml (faster and production grade vs xml.etree)
from lxml import etree

## 2. Paths & Configuration

In [221]:
# ── Paths ──────────────────────────────────────────────────────
INPUT_FOLDER  = Path("../data/raw")
OUTPUT_FOLDER = Path("../data/parsed/gen2")

# ── Config ─────────────────────────────────────────────────────
PREFERRED_REPORT_TYPE = "Consolidated"  # Consolidated | Standalone
SAVE_JSON             = True            # Save parsed JSON to disk

# ── Setup ───────────────────────────────────────────────────────
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

# ── Discover XML Files ──────────────────────────────────────────
xml_files = sorted(INPUT_FOLDER.glob("*.xml"))

# ── Summary ─────────────────────────────────────────────────────
print(f"Input  folder : {INPUT_FOLDER.resolve()}")
print(f"Output folder : {OUTPUT_FOLDER.resolve()}")
print(f"Report type   : {PREFERRED_REPORT_TYPE}")
print(f"XML files found: {len(xml_files)}")
print()
for f in xml_files:
    print(f"  - {f.name}")

Input  folder : C:\Users\Home\llmops-nse-rag\data\raw
Output folder : C:\Users\Home\llmops-nse-rag\data\parsed\gen2
Report type   : Consolidated
XML files found: 6

  - GI_117338_1350247_17012025074725.xml
  - INDAS_118123_1365717_30012025030002.xml
  - INTEGRATED_FILING_BANKING_1603568_17012026092624_WEB.xml
  - LI_117242_1346294_15012025050606.xml
  - NBFC_INDAS_118212_1366575_30012025073535.xml
  - RELIANCE_1425445_25042025095716_WEB.xml


## 3. Format Generation Detection
Detects whether XML file is Gen1 (BSE, pre-2025) or Gen2 (SEBI, post-2025)
by reading XML comment and namespace URL at top of file.

In [222]:
def detect_format_generation(file_path: Path) -> str:
    """
    Detects XBRL format generation from XML comment and namespace URL.

    Gen2 Indicators:
        - Comment : IFFIndAs | IFBanking | IFNBFC | IFInsurance
        - URL     : sebi.gov.in/xbrl/

    Gen1 Indicators:
        - No XML comment
        - URL     : bseindia.com/xbrl/

    Returns:
        "GEN2" or "GEN1"
    """

    # ── Gen2 comment indicators ─────────────────────────────────
    GEN2_COMMENTS = ["IFFIndAs", "IFBanking", "IFNBFC", "IFInsurance"]
    GEN2_URL      = "sebi.gov.in/xbrl"
    GEN1_URL      = "bseindia.com/xbrl"

    try:
        # Read only first 5 lines — no need to parse full file
        with file_path.open("r", encoding="utf-8") as f:
            head = "".join(f.readline() for _ in range(5))

        # ── Check Gen2 comment first (most reliable) ────────────
        for comment in GEN2_COMMENTS:
            if comment in head:
                return "GEN2"

        # ── Check namespace URL as backup ───────────────────────
        if GEN2_URL in head:
            return "GEN2"

        if GEN1_URL in head:
            return "GEN1"

        # ── Unknown format ───────────────────────────────────────
        return "UNKNOWN"

    except Exception as e:
        print(f"  [ERROR] Generation detection failed for {file_path.name}: {e}")
        return "UNKNOWN"


# ── Test on all files ───────────────────────────────────────────
print("Format Generation Detection Results:")
print("-" * 50)

for file_path in xml_files:
    generation = detect_format_generation(file_path)
    print(f"  {file_path.name:<50} → {generation}")

Format Generation Detection Results:
--------------------------------------------------
  GI_117338_1350247_17012025074725.xml               → GEN1
  INDAS_118123_1365717_30012025030002.xml            → GEN1
  INTEGRATED_FILING_BANKING_1603568_17012026092624_WEB.xml → GEN2
  LI_117242_1346294_15012025050606.xml               → GEN1
  NBFC_INDAS_118212_1366575_30012025073535.xml       → GEN1
  RELIANCE_1425445_25042025095716_WEB.xml            → GEN2


## 4. Taxonomy Detection
Detects company taxonomy (COMMERCIAL/BANK/NBFC/INSURANCE) from
namespace URL in XML file. Only applied to Gen2 files.

In [223]:
def detect_taxonomy(file_path: Path, generation: str) -> str:

    # ── Gen1 — skip taxonomy detection for now ──────────────────
    if generation == "GEN1":
        return "RAW_GEN1"

    # ── Gen2 taxonomy URL mapping ────────────────────────────────
    GEN2_TAXONOMY_MAP = {
        "IntegratedFinance_IndAS"     : "COMMERCIAL",
        "IntegratedFinance_Banking"   : "BANK",
        "IntegratedFinance_NBFC"      : "NBFC",
        "IntegratedFinance_Insurance" : "INSURANCE",
    }

    try:
        # ── Read first 5 lines only ──────────────────────────────
        with file_path.open("r", encoding="utf-8") as f:
            head = "".join(f.readline() for _ in range(5))

        # ── Match taxonomy from URL ──────────────────────────────
        for url_keyword, taxonomy in GEN2_TAXONOMY_MAP.items():
            if url_keyword in head:
                return taxonomy

        # ── Gen2 file but unknown taxonomy ───────────────────────
        return "UNKNOWN"

    except Exception as e:
        print(f"  [ERROR] Taxonomy detection failed for {file_path.name}: {e}")
        return "UNKNOWN"


# ── Test on all files ────────────────────────────────────────────
print("Taxonomy Detection Results:")
print("-" * 60)

gen2_count     = 0
gen1_count     = 0
unknown_count  = 0

for file_path in xml_files:

    generation = detect_format_generation(file_path)
    taxonomy   = detect_taxonomy(file_path, generation)

    print(f"  {file_path.name:<50} → {generation:<6} | {taxonomy}")

    if generation == "GEN2":
        gen2_count += 1
    elif generation == "GEN1":
        gen1_count += 1
    else:
        unknown_count += 1

# ── Summary ──────────────────────────────────────────────────────
print()
print(f"  Gen2 files   : {gen2_count}")
print(f"  Gen1 files   : {gen1_count}")
print(f"  Unknown files: {unknown_count}")
print(f"  Total        : {len(xml_files)}")

Taxonomy Detection Results:
------------------------------------------------------------
  GI_117338_1350247_17012025074725.xml               → GEN1   | RAW_GEN1
  INDAS_118123_1365717_30012025030002.xml            → GEN1   | RAW_GEN1
  INTEGRATED_FILING_BANKING_1603568_17012026092624_WEB.xml → GEN2   | BANK
  LI_117242_1346294_15012025050606.xml               → GEN1   | RAW_GEN1
  NBFC_INDAS_118212_1366575_30012025073535.xml       → GEN1   | RAW_GEN1
  RELIANCE_1425445_25042025095716_WEB.xml            → GEN2   | COMMERCIAL

  Gen2 files   : 2
  Gen1 files   : 4
  Unknown files: 0
  Total        : 6


## 5. Load & Parse XML with lxml

In [224]:
def load_xml(file_path: Path) -> Tuple[etree._Element, Dict[str, str]]:
    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    try:
        tree = etree.parse(str(file_path))
        root = tree.getroot()

        namespace_map = {
            prefix: uri
            for prefix, uri in root.nsmap.items()
            if prefix is not None
        }

        return root, namespace_map

    except etree.XMLSyntaxError as e:
        raise etree.XMLSyntaxError(f"Invalid XML {file_path.name}: {e}")


# ── Test on one Gen2 sample file ─────────────────────────────────
sample_file          = xml_files[0]
sample_root, sample_ns = load_xml(sample_file)

print(f"File          : {sample_file.name}")
print(f"Root tag      : {sample_root.tag}")
print(f"Total children: {len(list(sample_root))}")
print()
print("Namespaces detected:")
for prefix, uri in sample_ns.items():
    print(f"  {prefix:<20} → {uri}")

File          : GI_117338_1350247_17012025074725.xml
Root tag      : {http://www.xbrl.org/2003/instance}xbrl
Total children: 472

Namespaces detected:
  in-capmkt            → http://www.bseindia.com/xbrl/2020-03-31/in-capmkt
  in-capmkt-ent        → http://www.bseindia.com/xbrl/GeneralInsurance/2020-03-31/in-capmkt/in-capmkt-ent
  in-capmkt-roles      → http://www.bseindia.com/xbrl/GeneralInsurance/2020-03-31/in-capmkt-roles
  xbrldt               → http://xbrl.org/2005/xbrldt
  nonnum               → http://www.xbrl.org/dtr/type/non-numeric
  link                 → http://www.xbrl.org/2003/linkbase
  net                  → http://www.xbrl.org/2009/role/net
  num                  → http://www.xbrl.org/dtr/type/numeric
  xlink                → http://www.w3.org/1999/xlink
  iso4217              → http://www.xbrl.org/2003/iso4217
  negated              → http://www.xbrl.org/2009/role/negated
  xbrldi               → http://xbrl.org/2006/xbrldi
  xbrli                → http://www.xbrl.or

## 6. Extract Filing Metadata

In [240]:
# Tags to extract from XML — verified from real Reliance XML
METADATA_TAGS = {
    "company_name" : [
        "NameOfTheCompany",
        "NameOfBank",
    ],
    "report_type"  : [
        "NatureOfReportStandaloneConsolidated",
    ],
    "currency"     : [
        "DescriptionOfPresentationCurrency",
    ],
    "rounding"     : [
        "LevelOfRounding",
    ],
    "period_type"  : [
        "TypeOfReportingPeriod",
    ],
    "quarter"      : [
        "ReportingQuarter",
    ],
    "fy_start"     : [
        "DateOfStartOfFinancialYear",
    ],
    "fy_end"       : [
        "DateOfEndOfFinancialYear",
    ],
    "scrip_code"   : [
        "identifier",
    ],
}

ROUNDING_MAP = {
    "crores"  : 1,
    "lakhs"   : 0.01,
    "millions": 0.1,
    "actual"  : 0.0000001,
}

QUARTER_MAP = {
    "first quarter"  : "Q1",
    "second quarter" : "Q2",
    "third quarter"  : "Q3",
    "fourth quarter" : "Q4",
}


def extract_filing_metadata(
    root          : etree._Element,
    namespace_map : Dict[str, str],
) -> Dict[str, Any]:

    metadata  = {key: None for key in METADATA_TAGS}
    capmkt_ns = namespace_map.get("in-capmkt", "")
    xbrli_ns  = namespace_map.get("xbrli", "")

    # ── Extract scrip code from identifier ──────────────────────
    identifier_tag  = f"{{{xbrli_ns}}}identifier"
    identifier_elem = root.find(f".//{identifier_tag}")
    if identifier_elem is not None and identifier_elem.text:
        metadata["scrip_code"] = identifier_elem.text.strip()

    # ── Extract all other fields ─────────────────────────────────
    for field, candidate_tags in METADATA_TAGS.items():

        if field == "scrip_code":
            continue

        # Try each candidate tag — first match wins!
        for tag_name in candidate_tags:
            full_tag = f"{{{capmkt_ns}}}{tag_name}"
            element  = root.find(f".//{full_tag}")

            if element is not None and element.text:
                metadata[field] = element.text.strip()
                break  # First match found — stop looking!

    # ── Normalize rounding ───────────────────────────────────────
    if metadata["rounding"]:
        metadata["rounding_divisor"] = ROUNDING_MAP.get(
            metadata["rounding"].lower(), 1
        )
    else:
        metadata["rounding_divisor"] = 1

    # ── Normalize quarter ────────────────────────────────────────
    if metadata["quarter"]:
        metadata["quarter"] = QUARTER_MAP.get(
            metadata["quarter"].lower(),
            metadata["quarter"]
        )

    # ── Detect fiscal year ───────────────────────────────────────
    if metadata["fy_end"]:
        metadata["fiscal_year"] = f"FY{metadata['fy_end'][:4]}"
    else:
        metadata["fiscal_year"] = None

    return metadata


# ── Find first Gen2 file for testing ─────────────────────────────
gen2_sample = None
for file_path in xml_files:
    if detect_format_generation(file_path) == "GEN2":
        # if "RELIANCE" in file_path.name.upper():
            gen2_sample = file_path
            break

if gen2_sample is None:
    raise ValueError("No Gen2 files found!")

# ── Test on Gen2 sample ──────────────────────────────────────────
sample_root, sample_ns = load_xml(gen2_sample)
sample_metadata        = extract_filing_metadata(sample_root, sample_ns)

print(f"Testing on : {gen2_sample.name}")
print("-" * 50)
for key, value in sample_metadata.items():
    print(f"  {key:<20} : {value}")

Testing on : INTEGRATED_FILING_BANKING_1603568_17012026092624_WEB.xml
--------------------------------------------------
  company_name         : HDFC Bank Limited
  report_type          : Consolidated
  currency             : INR
  rounding             : Crores
  period_type          : Quarterly
  quarter              : Q3
  fy_start             : 2025-04-01
  fy_end               : 2026-03-31
  scrip_code           : 500180
  rounding_divisor     : 1
  fiscal_year          : FY2026


## 7. Build Context Dictionary
Extracts and classifies all contexts from XML into:
- SIMPLE  → Main financial contexts (OneD, FourD, OneI, PY_I)
- SEGMENT → Segment breakdown contexts (OneReportable)
- EXPENSE → Expense breakdown contexts (OneExpenses)
- OCI     → Other Comprehensive Income contexts (D_Items)
- SKIP    → Auditor/irrelevant contexts

In [241]:
def classify_context(context_id: str, has_scenario: bool) -> str:

    ctx = context_id.lower()

    if not has_scenario:
        return "SIMPLE"

    if "auditor" in ctx:
        return "SKIP"

    if "reportable" in ctx:
        return "SEGMENT"

    if "expense" in ctx:
        return "EXPENSE"

    if ctx.startswith("d_") and "auditor" not in ctx:
        return "OCI"

    return "SKIP"


def build_context_dictionary(
    root          : etree._Element,
    namespace_map : Dict[str, str],
) -> Tuple[Dict[str, Dict[str, Any]], int]:

    xbrli_ns     = namespace_map.get("xbrli", "")
    context_tag  = f"{{{xbrli_ns}}}context"
    period_tag   = f"{{{xbrli_ns}}}period"
    start_tag    = f"{{{xbrli_ns}}}startDate"
    end_tag      = f"{{{xbrli_ns}}}endDate"
    instant_tag  = f"{{{xbrli_ns}}}instant"
    scenario_tag = f"{{{xbrli_ns}}}scenario"

    contexts    = {}
    total_count = 0

    for ctx in root.findall(f".//{context_tag}"):

        context_id = ctx.get("id")
        if not context_id:
            continue

        total_count  += 1
        has_scenario  = ctx.find(scenario_tag) is not None
        category      = classify_context(context_id, has_scenario)

        if category == "SKIP":
            continue

        period_node  = ctx.find(period_tag)
        if period_node is None:
            continue

        start_date   = period_node.findtext(start_tag)
        end_date     = period_node.findtext(end_tag)
        instant_date = period_node.findtext(instant_tag)

        if start_date and end_date:
            period_type = "duration"
        elif instant_date:
            period_type = "instant"
        else:
            continue

        contexts[context_id] = {
            "context_id"      : context_id,
            "context_type"    : period_type,
            "context_category": category,
            "start_date"      : start_date.strip()   if start_date   else None,
            "end_date"        : end_date.strip()     if end_date     else None,
            "instant_date"    : instant_date.strip() if instant_date else None,
        }

    return contexts, total_count


# ── Test on Reliance Gen2 sample ─────────────────────────────────
sample_contexts, total_count = build_context_dictionary(
    sample_root, sample_ns
)

# ── Count by category ────────────────────────────────────────────
category_counts = {
    "SIMPLE" : 0,
    "SEGMENT": 0,
    "EXPENSE": 0,
    "OCI"    : 0,
}

for ctx in sample_contexts.values():
    cat = ctx["context_category"]
    if cat in category_counts:
        category_counts[cat] += 1

skipped = total_count - len(sample_contexts)

# ── Summary ──────────────────────────────────────────────────────
print(f"Total contexts in file : {total_count}")
print(f"Simple contexts        : {category_counts['SIMPLE']}")
print(f"Segment contexts       : {category_counts['SEGMENT']}")
print(f"Expense contexts       : {category_counts['EXPENSE']}")
print(f"OCI contexts           : {category_counts['OCI']}")
print(f"Skipped contexts       : {skipped}")
print(f"Total extracted        : {len(sample_contexts)}")
print()

# ── Show extracted contexts ──────────────────────────────────────
print("Extracted Contexts:")
print("-" * 90)
for ctx_id, ctx_data in sample_contexts.items():
    print(f"  {ctx_id:<45} → "
          f"category: {ctx_data['context_category']:<10} | "
          f"type: {ctx_data['context_type']:<10} | "
          f"start: {ctx_data['start_date']} | "
          f"end: {ctx_data['end_date']} | "
          f"instant: {ctx_data['instant_date']}")

Total contexts in file : 57
Simple contexts        : 3
Segment contexts       : 48
Expense contexts       : 4
OCI contexts           : 0
Skipped contexts       : 2
Total extracted        : 55

Extracted Contexts:
------------------------------------------------------------------------------------------
  OneD                                          → category: SIMPLE     | type: duration   | start: 2025-10-01 | end: 2025-12-31 | instant: None
  OneI                                          → category: SIMPLE     | type: instant    | start: None | end: None | instant: 2025-12-31
  FourD                                         → category: SIMPLE     | type: duration   | start: 2025-04-01 | end: 2025-12-31 | instant: None
  OneExpenses1D                                 → category: EXPENSE    | type: duration   | start: 2025-10-01 | end: 2025-12-31 | instant: None
  OneExpenses2D                                 → category: EXPENSE    | type: duration   | start: 2025-10-01 | end: 2025-12-3

## 8. Extract Financial Facts
Dynamically identifies and extracts:
- Numeric facts  → decimals attribute present (including INF)
- Text blocks    → rich financial notes and disclosures
No hardcoded skip tags — works for any company or taxonomy!

In [242]:
def clean_text_block(raw_text: Optional[str]) -> Optional[str]:
    """
    Cleans HTML tags from text blocks.
    Converts <BR> to newlines and removes other HTML tags.
    """
    if not raw_text:
        return None

    # Replace <BR> variants with newline
    text = re.sub(r"<BR\s*/?>", "\n", raw_text, flags=re.IGNORECASE)

    # Remove remaining HTML tags
    text = re.sub(r"<[^>]+>", "", text)

    # Clean extra whitespace
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r" {2,}", " ", text)

    return text.strip()


def is_financial_fact(decimals: Optional[str]) -> bool:
    """
    Determines if tag is a numeric financial fact.
    Includes INF decimals — EPS, ratios, face value etc.
    """
    if decimals is None:
        return False
    return True


def normalize_value(
    raw_value : Optional[str],
    decimals  : Optional[str],
) -> Optional[float]:
    """
    Converts raw XML value to actual value.

    decimals = "-7"  → divide by 10^7
    decimals = "INF" → exact value, no conversion
    decimals = "0"   → no conversion
    """
    if raw_value is None:
        return None

    try:
        numeric = float(raw_value.strip().replace(",", ""))
    except ValueError:
        return None

    if decimals is None:
        return numeric

    # INF means exact value — no conversion needed
    if decimals.strip().upper() == "INF":
        return numeric

    try:
        decimal_int = int(decimals)
        if decimal_int < 0:
            return numeric / (10 ** abs(decimal_int))
        return numeric
    except ValueError:
        return numeric
    
def is_text_block(
    tag_name : str,
    decimals : Optional[str],
    text     : Optional[str],
) -> bool:

    if decimals is not None:
        return False

    if not text or len(text.strip()) < 200:
        return False

    FINANCIAL_KEYWORDS = [
        "ratio", "revenue", "profit", "loss",
        "margin", "earnings", "income", "expense",
        "segment", "financial", "result", "crore",
        "quarter", "annual", "fiscal", "interest",
        "capital", "assets", "liabilities", "tax",
    ]

    text_lower  = text.lower()
    has_keyword = any(
        keyword in text_lower
        for keyword in FINANCIAL_KEYWORDS
    )

    return has_keyword


def extract_financial_facts(
    root          : etree._Element,
    namespace_map : Dict[str, str],
    contexts      : Dict[str, Dict[str, Any]],
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    """
    Returns:
        numeric_facts → list of numeric financial facts
        text_blocks   → list of rich text disclosures
    """

    capmkt_ns    = namespace_map.get("in-capmkt", "")
    numeric_facts = []
    text_blocks   = []

    for elem in root.iter():

        if not isinstance(elem.tag, str):
            continue
        if capmkt_ns not in elem.tag:
            continue

        tag_name    = elem.tag.split("}")[-1]
        context_ref = elem.get("contextRef")
        decimals    = elem.get("decimals")

        # Skip if no context or not in our dictionary
        if not context_ref or context_ref not in contexts:
            continue

        # Skip empty values
        if not elem.text or not elem.text.strip():
            continue

        raw_value = elem.text.strip()

        # ── Extract text blocks ──────────────────────────────────
        if is_text_block(tag_name, decimals, raw_value):
            cleaned_text = clean_text_block(raw_value)
            if cleaned_text:
                text_blocks.append({
                    "tag_name"        : tag_name,
                    "context_id"      : context_ref,
                    "context_category": contexts[context_ref]["context_category"],
                    "context_type"    : contexts[context_ref]["context_type"],
                    "start_date"      : contexts[context_ref]["start_date"],
                    "end_date"        : contexts[context_ref]["end_date"],
                    "instant_date"    : contexts[context_ref]["instant_date"],
                    "text"            : cleaned_text,
                })
            continue

        # ── Extract numeric facts ────────────────────────────────
        if not is_financial_fact(decimals):
            continue

        norm_value = normalize_value(raw_value, decimals)

        numeric_facts.append({
            "tag_name"        : tag_name,
            "context_id"      : context_ref,
            "context_category": contexts[context_ref]["context_category"],
            "context_type"    : contexts[context_ref]["context_type"],
            "start_date"      : contexts[context_ref]["start_date"],
            "end_date"        : contexts[context_ref]["end_date"],
            "instant_date"    : contexts[context_ref]["instant_date"],
            "raw_value"       : raw_value,
            "decimals"        : decimals,
            "normalized_value": norm_value,
            "is_numeric"      : norm_value is not None,
            "is_inf"          : decimals.strip().upper() == "INF",
        })

    return numeric_facts, text_blocks


# ── Test on Reliance Gen2 sample ─────────────────────────────────
sample_facts, sample_text_blocks = extract_financial_facts(
    root          = sample_root,
    namespace_map = sample_ns,
    contexts      = sample_contexts,
)

# ── Numeric facts summary ────────────────────────────────────────
simple_facts  = [f for f in sample_facts if f["context_category"] == "SIMPLE"]
segment_facts = [f for f in sample_facts if f["context_category"] == "SEGMENT"]
expense_facts = [f for f in sample_facts if f["context_category"] == "EXPENSE"]
oci_facts     = [f for f in sample_facts if f["context_category"] == "OCI"]
inf_facts     = [f for f in sample_facts if f["is_inf"]]

print(f"Total numeric facts : {len(sample_facts)}")
print(f"Simple facts        : {len(simple_facts)}")
print(f"Segment facts       : {len(segment_facts)}")
print(f"Expense facts       : {len(expense_facts)}")
print(f"OCI facts           : {len(oci_facts)}")
print(f"INF facts (EPS etc) : {len(inf_facts)}")
print(f"Text blocks         : {len(sample_text_blocks)}")
print()

# ── Show INF facts ───────────────────────────────────────────────
print("INF Facts (EPS, Ratios, Face Value):")
print("-" * 70)
for fact in inf_facts:
    print(f"  {fact['tag_name']:<60} : {fact['normalized_value']}")

print()

# ── Show text blocks ─────────────────────────────────────────────
print("Text Blocks Extracted:")
print("-" * 70)
for tb in sample_text_blocks:
    print(f"  Tag     : {tb['tag_name']}")
    print(f"  Context : {tb['context_id']}")
    print(f"  Preview : {tb['text'][:200]}...")
    print()

Total numeric facts : 134
Simple facts        : 94
Segment facts       : 36
Expense facts       : 4
OCI facts           : 0
INF facts (EPS etc) : 22
Text blocks         : 1

INF Facts (EPS, Ratios, Face Value):
----------------------------------------------------------------------
  FaceValueOfEquityShareCapital                                : 1.0
  PercentageOfShareHeldByGovernmentOfIndia                     : 0.0
  CET1Ratio                                                    : 0.0
  AdditionalTier1Ratio                                         : 0.0
  BasicEarningsPerShareBeforeExtraordinaryItems                : 12.88
  DilutedEarningsPerShareBeforeExtraordinaryItems              : 12.82
  BasicEarningsPerShareAfterExtraordinaryItems                 : 12.88
  DilutedEarningsPerShareAfterExtraordinaryItems               : 12.82
  PercentageOfGrossNpa                                         : 0.0
  PercentageOfNpa                                              : 0.0
  ReturnOnAssets    

## 9. Build Lean JSON
Organizes all extracted data into lean structured JSON.
Facts grouped by context/period + text blocks separately.
Ready for chunking pipeline.

In [243]:
def build_lean_json(
    source_file   : Path,
    taxonomy      : str,
    generation    : str,
    metadata      : Dict[str, Any],
    contexts      : Dict[str, Dict[str, Any]],
    facts         : List[Dict[str, Any]],
    text_blocks   : List[Dict[str, Any]],
) -> Dict[str, Any]:

    # ── Group numeric facts by context_id ────────────────────────
    facts_by_context = defaultdict(list)
    for fact in facts:
        facts_by_context[fact["context_id"]].append(fact)

    # ── Build periods list ───────────────────────────────────────
    periods = []

    for context_id, context_data in contexts.items():

        context_facts = facts_by_context.get(context_id, [])

        # Skip contexts with no facts
        if not context_facts:
            continue

        # Build facts dictionary
        numeric_facts = {}
        inf_facts     = {}

        for fact in context_facts:
            if fact["is_inf"]:
                inf_facts[fact["tag_name"]]     = fact["normalized_value"]
            else:
                numeric_facts[fact["tag_name"]] = fact["normalized_value"]

        periods.append({
            "context_id"      : context_id,
            "context_category": context_data["context_category"],
            "context_type"    : context_data["context_type"],
            "start_date"      : context_data["start_date"],
            "end_date"        : context_data["end_date"],
            "instant_date"    : context_data["instant_date"],
            "numeric_facts"   : numeric_facts,
            "inf_facts"       : inf_facts,
            "fact_count"      : len(context_facts),
        })

    # ── Sort periods — SIMPLE first then others ──────────────────
    category_order = {
        "SIMPLE" : 0,
        "OCI"    : 1,
        "EXPENSE": 2,
        "SEGMENT": 3,
    }
    periods.sort(
        key=lambda x: category_order.get(x["context_category"], 99)
    )

    # ── Build text blocks list ───────────────────────────────────
    clean_text_blocks = []
    for tb in text_blocks:
        clean_text_blocks.append({
            "tag_name"        : tb["tag_name"],
            "context_id"      : tb["context_id"],
            "context_category": tb["context_category"],
            "start_date"      : tb["start_date"],
            "end_date"        : tb["end_date"],
            "instant_date"    : tb["instant_date"],
            "text"            : tb["text"],
        })

    # ── Build final lean JSON ────────────────────────────────────
    lean_json = {

        # File info
        "source_file"      : source_file.name,
        "format_generation": generation,

        # Company info
        "company_name"     : metadata.get("company_name"),
        "scrip_code"       : metadata.get("scrip_code"),
        "taxonomy"         : taxonomy,

        # Filing info
        "report_type"      : metadata.get("report_type"),
        "currency"         : metadata.get("currency"),
        "rounding"         : metadata.get("rounding"),
        "rounding_divisor" : metadata.get("rounding_divisor"),

        # Period info
        "period_type"      : metadata.get("period_type"),
        "quarter"          : metadata.get("quarter"),
        "fiscal_year"      : metadata.get("fiscal_year"),
        "fy_start"         : metadata.get("fy_start"),
        "fy_end"           : metadata.get("fy_end"),

        # Stats
        "total_periods"    : len(periods),
        "total_facts"      : len(facts),
        "total_text_blocks": len(clean_text_blocks),

        # Data
        "periods"          : periods,
        "text_blocks"      : clean_text_blocks,
    }

    return lean_json


# ── Test on Reliance Gen2 sample ─────────────────────────────────
sample_lean_json = build_lean_json(
    source_file = gen2_sample,
    taxonomy    = detect_taxonomy(gen2_sample, "GEN2"),
    generation  = "GEN2",
    metadata    = sample_metadata,
    contexts    = sample_contexts,
    facts       = sample_facts,
    text_blocks = sample_text_blocks,
)

# ── Summary ──────────────────────────────────────────────────────
print(f"Company          : {sample_lean_json['company_name']}")
print(f"Scrip Code       : {sample_lean_json['scrip_code']}")
print(f"Taxonomy         : {sample_lean_json['taxonomy']}")
print(f"Report Type      : {sample_lean_json['report_type']}")
print(f"Currency         : {sample_lean_json['currency']}")
print(f"Rounding         : {sample_lean_json['rounding']}")
print(f"Period Type      : {sample_lean_json['period_type']}")
print(f"Quarter          : {sample_lean_json['quarter']}")
print(f"Fiscal Year      : {sample_lean_json['fiscal_year']}")
print(f"Total Periods    : {sample_lean_json['total_periods']}")
print(f"Total Facts      : {sample_lean_json['total_facts']}")
print(f"Total Text Blocks: {sample_lean_json['total_text_blocks']}")
print()

# ── Periods summary ──────────────────────────────────────────────
print("Periods Summary:")
print("-" * 90)
for period in sample_lean_json["periods"]:
    print(f"  {period['context_id']:<45} → "
          f"category: {period['context_category']:<10} | "
          f"facts: {period['fact_count']:<5} | "
          f"start: {period['start_date']} | "
          f"end: {period['end_date']} | "
          f"instant: {period['instant_date']}")

print()

# ── Show OneD numeric facts ──────────────────────────────────────
print("Sample Numeric Facts from OneD:")
print("-" * 70)
one_d = next(
    (p for p in sample_lean_json["periods"]
     if p["context_id"] == "OneD"),
    None
)
if one_d:
    for tag, value in list(one_d["numeric_facts"].items())[:10]:
        print(f"  {tag:<60} : {value}")

print()

# ── Show OneD INF facts ──────────────────────────────────────────
print("INF Facts from OneD (EPS, Ratios):")
print("-" * 70)
if one_d:
    for tag, value in one_d["inf_facts"].items():
        print(f"  {tag:<60} : {value}")

print()

# ── Show text blocks ─────────────────────────────────────────────
print("Text Blocks:")
print("-" * 70)
for tb in sample_lean_json["text_blocks"]:
    print(f"  Tag    : {tb['tag_name']}")
    print(f"  Context: {tb['context_id']}")
    print(f"  Preview: {tb['text'][:150]}...")
    print()

Company          : HDFC Bank Limited
Scrip Code       : 500180
Taxonomy         : BANK
Report Type      : Consolidated
Currency         : INR
Rounding         : Crores
Period Type      : Quarterly
Quarter          : Q3
Fiscal Year      : FY2026
Total Periods    : 43
Total Facts      : 134
Total Text Blocks: 1

Periods Summary:
------------------------------------------------------------------------------------------
  OneD                                          → category: SIMPLE     | facts: 44    | start: 2025-10-01 | end: 2025-12-31 | instant: None
  OneI                                          → category: SIMPLE     | facts: 6     | start: None | end: None | instant: 2025-12-31
  FourD                                         → category: SIMPLE     | facts: 44    | start: 2025-04-01 | end: 2025-12-31 | instant: None
  OneExpenses1D                                 → category: EXPENSE    | facts: 1     | start: 2025-10-01 | end: 2025-12-31 | instant: None
  OneExpenses2D           

## 10. Validate Lean JSON
Performs comprehensive validation on the generated lean JSON
to ensure data quality, structure, and completeness before saving.

In [244]:
def validate_lean_json(lean_json: Dict[str, Any]) -> Dict[str, Any]:
    """
    Validates the lean JSON structure and data quality.
    Returns detailed validation report.
    """

    validation = {
        "is_valid"      : True,
        "status"        : "PASSED",
        "errors"        : [],
        "warnings"      : [],
        "stats"         : {}
    }

    # ── 1. Structure Validation ──────────────────────────────────
    required_keys = [
        "company_name", "scrip_code", "taxonomy", "format_generation", "fiscal_year", "currency",
        "rounding", "periods", "text_blocks", "total_periods", "total_facts", "total_text_blocks",
    ]

    for key in required_keys:
        if key not in lean_json:
            validation["errors"].append(f"Missing required key: '{key}'")
            validation["is_valid"] = False

    # ── 2. Data Quality Checks ───────────────────────────────────
    if not lean_json.get("company_name"):
        validation["errors"].append("company_name is missing or empty")

    if not lean_json.get("scrip_code"):
        validation["errors"].append("scrip_code is missing or empty")

    if lean_json.get("taxonomy") not in ["COMMERCIAL", "BANK", "NBFC", "INSURANCE"]:
        validation["errors"].append(
            f"Invalid taxonomy: {lean_json.get('taxonomy')}"
        )

    if not lean_json.get("fiscal_year"):
        validation["errors"].append("fiscal_year is missing or empty")

    if not lean_json.get("currency"):
        validation["errors"].append("currency is missing or empty")

    if not lean_json.get("rounding"):
        validation["errors"].append("rounding is missing or empty")

    if lean_json.get("total_facts", 0) <= 0:
        validation["errors"].append("total_facts is 0 — no facts extracted!")

    if not lean_json.get("periods"):
        validation["errors"].append("No periods found in data")

    # Warning — not critical but important
    if not lean_json.get("quarter"):
        if lean_json.get("period_type", "").lower() == "quarterly":
            validation["warnings"].append(
                "quarter is missing but period_type is Quarterly!"
            )

    # ── 3. Fact Count Consistency ────────────────────────────────
    if lean_json.get("periods"):
        calculated_facts = sum(
            p.get("fact_count", 0) 
            for p in lean_json["periods"]
        )

    if calculated_facts != lean_json.get("total_facts", 0):
        validation["warnings"].append(
            f"Fact count mismatch: "
            f"periods={calculated_facts}, "
            f"total_facts={lean_json['total_facts']}"
        )

    # ── 4. Empty Periods Check ───────────────────────────────────
    empty_periods = [
        p["context_id"] for p in lean_json["periods"]
        if not p.get("numeric_facts") and not p.get("inf_facts")
    ]

    if empty_periods:
        validation["warnings"].append(
            f"Periods with no facts: {empty_periods}"
        )

    # ── 5. Text Block Quality Check ──────────────────────────────
    for tb in lean_json["text_blocks"]:
        if len(tb.get("text", "")) < 100:
            validation["warnings"].append(
                f"Text block too short: {tb.get('tag_name')} "
                f"({len(tb.get('text', ''))} chars)"
            )

    # ── 6. Stats Summary ─────────────────────────────────────────
    validation["stats"] = {
        "company_name"      : lean_json.get("company_name"),
        "taxonomy"          : lean_json.get("taxonomy"),
        "total_periods"     : len(lean_json["periods"]),
        "simple_periods"    : sum(1 for p in lean_json["periods"] 
                                 if p["context_category"] == "SIMPLE"),
        "segment_periods"   : sum(1 for p in lean_json["periods"] 
                                 if p["context_category"] == "SEGMENT"),
        "oci_periods"       : sum(1 for p in lean_json["periods"] 
                                 if p["context_category"] == "OCI"),
        "total_numeric_facts": lean_json["total_facts"],
        "total_text_blocks" : len(lean_json["text_blocks"]),
    }

    # ── Final Status ─────────────────────────────────────────────
    if validation["errors"]:
        validation["status"] = "FAILED"
        validation["is_valid"] = False
    elif validation["warnings"]:
        validation["status"] = "WARNING"
    else:
        validation["status"] = "PASSED"

    return validation


# ── Run Validation ───────────────────────────────────────────────
print("Running Lean JSON Validation...")
print("=" * 70)

validation_result = validate_lean_json(sample_lean_json)

print(f"Validation Status : {validation_result['status']}")
print(f"Is Valid          : {validation_result['is_valid']}")
print()

print("Statistics:")
print("-" * 50)
for key, value in validation_result["stats"].items():
    print(f"  {key:<25} : {value}")

print()

if validation_result["errors"]:
    print("ERRORS:")
    for error in validation_result["errors"]:
        print(f"   {error}")

if validation_result["warnings"]:
    print("WARNINGS:")
    for warning in validation_result["warnings"]:
        print(f"    {warning}")

if validation_result["status"] == "PASSED":
    print("\n Lean JSON validation PASSED — Ready for chunking pipeline!")
elif validation_result["status"] == "WARNING":
    print("\n  Lean JSON passed with warnings — still usable")
else:
    print("\n Lean JSON validation FAILED — check errors above")

Running Lean JSON Validation...
Validation Status : PASSED
Is Valid          : True

Statistics:
--------------------------------------------------
  company_name              : HDFC Bank Limited
  taxonomy                  : BANK
  total_periods             : 43
  simple_periods            : 3
  segment_periods           : 36
  oci_periods               : 0
  total_numeric_facts       : 134
  total_text_blocks         : 1


 Lean JSON validation PASSED — Ready for chunking pipeline!


In [245]:
import traceback

try:
    sample_root, sample_ns = load_xml(gen2_sample)
    metadata = extract_filing_metadata(sample_root, sample_ns)
    print("Success!")
    for k, v in metadata.items():
        print(f"  {k:<20} : {v}")
        
except Exception as e:
    print(f"Error: {e}")
    traceback.print_exc()

Success!
  company_name         : HDFC Bank Limited
  report_type          : Consolidated
  currency             : INR
  rounding             : Crores
  period_type          : Quarterly
  quarter              : Q3
  fy_start             : 2025-04-01
  fy_end               : 2026-03-31
  scrip_code           : 500180
  rounding_divisor     : 1
  fiscal_year          : FY2026


## 11. Master Parse Function
Single entry point to parse any Gen2 XBRL file end-to-end.
Combines all steps: load → metadata → contexts → facts → lean JSON → validate.

In [246]:
def parse_xbrl_gen2(file_path: Path) -> Dict[str, Any]:
    """
    Master function to parse a single Gen2 XBRL file.
    
    Returns:
        Dict with keys:
            - lean_json        : parsed lean JSON
            - validation       : validation result
            - success          : True/False
            - error            : error message if failed
    """

    result = {
        "file_path"  : str(file_path),
        "file_name"  : file_path.name,
        "lean_json"  : None,
        "validation" : None,
        "success"    : False,
        "error"      : None,
    }

    try:
        # ── Step 1: Verify Gen2 format ───────────────────────────
        generation = detect_format_generation(file_path)
        if generation != "GEN2":
            raise ValueError(
                f"File is not Gen2 format: {generation}"
            )

        # ── Step 2: Detect taxonomy ──────────────────────────────
        taxonomy = detect_taxonomy(file_path, generation)
        if taxonomy == "UNKNOWN":
            raise ValueError(
                f"Could not detect taxonomy for: {file_path.name}"
            )

        # ── Step 3: Load XML ─────────────────────────────────────
        root, namespace_map = load_xml(file_path)

        # ── Step 4: Extract filing metadata ──────────────────────
        metadata = extract_filing_metadata(root, namespace_map)

        # ── Step 5: Filter by preferred report type ──────────────
        report_type = metadata.get("report_type", "")
        if (report_type and
            PREFERRED_REPORT_TYPE and
            report_type.lower() != PREFERRED_REPORT_TYPE.lower()):
            raise ValueError(
                f"Report type mismatch: "
                f"expected {PREFERRED_REPORT_TYPE}, "
                f"got {report_type}"
            )

        # ── Step 6: Build context dictionary ─────────────────────
        contexts, total_context_count = build_context_dictionary(
            root, namespace_map
        )

        if not contexts:
            raise ValueError("No valid contexts found in file!")

        # ── Step 7: Extract financial facts ──────────────────────
        facts, text_blocks = extract_financial_facts(
            root, namespace_map, contexts
        )

        if not facts:
            raise ValueError("No financial facts extracted!")

        # ── Step 8: Build lean JSON ───────────────────────────────
        lean_json = build_lean_json(
            source_file = file_path,
            taxonomy    = taxonomy,
            generation  = generation,
            metadata    = metadata,
            contexts    = contexts,
            facts       = facts,
            text_blocks = text_blocks,
        )

        # ── Step 9: Validate lean JSON ────────────────────────────
        validation = validate_lean_json(lean_json)

        if not validation["is_valid"]:
            raise ValueError(
                f"Validation failed: {validation['errors']}"
            )

        result["lean_json"]  = lean_json
        result["validation"] = validation
        result["success"]    = True

    except Exception as e:
        result["success"] = False
        result["error"]   = str(e)

    return result


# ── Test on Gen2 sample file ─────────────────────────────────────
print(f"Testing master parser on: {gen2_sample.name}")
print("=" * 70)

test_result = parse_xbrl_gen2(gen2_sample)

if test_result["success"]:
    lj = test_result["lean_json"]
    vl = test_result["validation"]

    print(f"Status           : SUCCESS ✅")
    print(f"Company          : {lj['company_name']}")
    print(f"Taxonomy         : {lj['taxonomy']}")
    print(f"Report Type      : {lj['report_type']}")
    print(f"Quarter          : {lj['quarter']}")
    print(f"Fiscal Year      : {lj['fiscal_year']}")
    print(f"Total Periods    : {lj['total_periods']}")
    print(f"Total Facts      : {lj['total_facts']}")
    print(f"Total Text Blocks: {lj['total_text_blocks']}")
    print(f"Validation       : {vl['status']}")

    if vl["warnings"]:
        print(f"\nWarnings:")
        for w in vl["warnings"]:
            print(f"  ⚠️  {w}")
else:
    print(f"Status : FAILED ❌")
    print(f"Error  : {test_result['error']}")

Testing master parser on: INTEGRATED_FILING_BANKING_1603568_17012026092624_WEB.xml
Status           : SUCCESS ✅
Company          : HDFC Bank Limited
Taxonomy         : BANK
Report Type      : Consolidated
Quarter          : Q3
Fiscal Year      : FY2026
Total Periods    : 43
Total Facts      : 134
Total Text Blocks: 1
Validation       : PASSED


## 12. Batch Processing
Processes all Gen2 XBRL files in input folder.
Skips Gen1 files automatically — handled separately later.
Saves each parsed lean JSON to output folder.

In [250]:
def process_batch(
    xml_files     : List[Path],
    output_folder : Path,
    save_json     : bool = True,
) -> Dict[str, Any]:

    batch_report = {
        "total_files"   : len(xml_files),
        "success"       : [],
        "failed"        : [],
        "skipped_gen1"  : [],
        "skipped_other" : [],
    }

    print(f"Starting batch processing...")
    print(f"Total files found : {len(xml_files)}")
    print(f"Output folder     : {output_folder}")
    print("=" * 70)

    for idx, file_path in enumerate(xml_files, 1):

        print(f"\n[{idx}/{len(xml_files)}] {file_path.name}")

        # ── Check generation ─────────────────────────────────────
        generation = detect_format_generation(file_path)

        if generation == "GEN1":
            print(f"    Skipped — Gen1 format")
            batch_report["skipped_gen1"].append(file_path.name)
            continue

        if generation == "UNKNOWN":
            print(f"    Skipped — Unknown format")
            batch_report["skipped_other"].append(file_path.name)
            continue

        # ── Parse Gen2 file ──────────────────────────────────────
        result = parse_xbrl_gen2(file_path)

        if result["success"]:
            lean_json  = result["lean_json"]
            validation = result["validation"]

            # ── Save JSON ────────────────────────────────────────
            if save_json:
                output_file = output_folder / (
                    file_path.stem + "_parsed.json"
                )
                with output_file.open("w", encoding="utf-8") as f:
                    json.dump(lean_json, f, indent=2, ensure_ascii=False)

            batch_report["success"].append({
                "file_name"        : file_path.name,
                "company_name"     : lean_json["company_name"],
                "taxonomy"         : lean_json["taxonomy"],
                "report_type"      : lean_json["report_type"],
                "quarter"          : lean_json["quarter"],
                "fiscal_year"      : lean_json["fiscal_year"],
                "total_periods"    : lean_json["total_periods"],
                "total_facts"      : lean_json["total_facts"],
                "total_text_blocks": lean_json["total_text_blocks"],
                "validation_status": validation["status"],
                "output_file"      : output_file.name if save_json else None,
            })

            print(f"   Success   | "
                  f"Company: {lean_json['company_name']:<45} | "
                  f"Taxonomy: {lean_json['taxonomy']:<12} | "
                  f"Quarter: {str(lean_json['quarter']):<4} | "
                  f"FY: {str(lean_json['fiscal_year']):<7} | "
                  f"Facts: {lean_json['total_facts']:<5} | "
                  f"Validation: {validation['status']}")

            if validation["warnings"]:
                for w in validation["warnings"]:
                    print(f"        {w}")

        else:
            batch_report["failed"].append({
                "file_name": file_path.name,
                "error"    : result["error"],
            })
            print(f"   Failed    | Error: {result['error']}")

    return batch_report


# ── Run Batch Processing ─────────────────────────────────────────
batch_report = process_batch(
    xml_files     = xml_files,
    output_folder = OUTPUT_FOLDER,
    save_json     = SAVE_JSON,
)

Starting batch processing...
Total files found : 6
Output folder     : ..\data\parsed\gen2

[1/6] GI_117338_1350247_17012025074725.xml
    Skipped — Gen1 format

[2/6] INDAS_118123_1365717_30012025030002.xml
    Skipped — Gen1 format

[3/6] INTEGRATED_FILING_BANKING_1603568_17012026092624_WEB.xml
   Success   | Company: HDFC Bank Limited                             | Taxonomy: BANK         | Quarter: Q3   | FY: FY2026  | Facts: 134   | Validation: PASSED

[4/6] LI_117242_1346294_15012025050606.xml
    Skipped — Gen1 format

[5/6] NBFC_INDAS_118212_1366575_30012025073535.xml
    Skipped — Gen1 format

[6/6] RELIANCE_1425445_25042025095716_WEB.xml
   Success   | Company: RELIANCE INDUSTRIES LIMITED                   | Taxonomy: COMMERCIAL   | Quarter: Q4   | FY: FY2025  | Facts: 302   | Validation: PASSED


## 13. Batch Summary
Final summary of batch processing results.
Shows success/failed/skipped counts and saves batch report.

In [252]:
def print_batch_summary(batch_report: Dict[str, Any]) -> None:

    success       = batch_report["success"]
    failed        = batch_report["failed"]
    skipped_gen1  = batch_report["skipped_gen1"]
    skipped_other = batch_report["skipped_other"]
    total         = batch_report["total_files"]

    print("=" * 70)
    print("BATCH PROCESSING SUMMARY")
    print("=" * 70)

    # ── File counts ──────────────────────────────────────────────
    print(f"\nFile Counts:")
    print(f"  Total files found  : {total}")
    print(f"  Successfully parsed: {len(success)}")
    print(f"  Failed             : {len(failed)}")
    print(f"  Skipped (Gen1)     : {len(skipped_gen1)}")
    print(f"  Skipped (Other)    : {len(skipped_other)}")

    # ── Success breakdown ────────────────────────────────────────
    if success:
        print(f"\nSuccessfully Parsed Files:")
        print("-" * 70)

        # Group by taxonomy
        taxonomy_groups = {}
        for s in success:
            tax = s["taxonomy"]
            if tax not in taxonomy_groups:
                taxonomy_groups[tax] = []
            taxonomy_groups[tax].append(s)

        for taxonomy, files in taxonomy_groups.items():
            print(f"\n  {taxonomy} ({len(files)} files):")
            for f in files:
                print(f"     {f['company_name']:<45} | "
                      f"Q: {str(f['quarter']):<4} | "
                      f"FY: {str(f['fiscal_year']):<7} | "
                      f"Facts: {f['total_facts']:<5} | "
                      f"Validation: {f['validation_status']}")

    # ── Failed files ─────────────────────────────────────────────
    if failed:
        print(f"\nFailed Files:")
        print("-" * 70)
        for f in failed:
            print(f"   {f['file_name']}")
            print(f"     Error: {f['error']}")

    # ── Skipped Gen1 files ───────────────────────────────────────
    if skipped_gen1:
        print(f"\nSkipped Gen1 Files (to be handled separately):")
        print("-" * 70)
        for f in skipped_gen1:
            print(f"    {f}")

    # ── Skipped other files ──────────────────────────────────────
    if skipped_other:
        print(f"\nSkipped Unknown Format Files:")
        print("-" * 70)
        for f in skipped_other:
            print(f"    {f}")

    # ── Overall stats ────────────────────────────────────────────
    if success:
        total_facts       = sum(s["total_facts"]       for s in success)
        total_periods     = sum(s["total_periods"]     for s in success)
        total_text_blocks = sum(s["total_text_blocks"] for s in success)

        print(f"\nOverall Statistics (Parsed Files):")
        print("-" * 70)
        print(f"  Total facts extracted      : {total_facts}")
        print(f"  Total periods extracted    : {total_periods}")
        print(f"  Total text blocks extracted: {total_text_blocks}")
        print(f"  Avg facts per file         : {total_facts // len(success)}")

    # ── Next steps ───────────────────────────────────────────────
    print(f"\nOutput Location:")
    print(f"  Parsed JSONs saved to: {OUTPUT_FOLDER.resolve()}")
    print(f"\nNext Step:")
    print(f"  → 02_chunking.ipynb")
    print(f"  → Load parsed JSONs from: {OUTPUT_FOLDER.resolve()}")


# ── Save batch report to JSON ────────────────────────────────────
def save_batch_report(
    batch_report  : Dict[str, Any],
    output_folder : Path,
) -> None:

    report_path = output_folder / "batch_report.json"

    # Convert to serializable format
    serializable_report = {
        "total_files"   : batch_report["total_files"],
        "success_count" : len(batch_report["success"]),
        "failed_count"  : len(batch_report["failed"]),
        "skipped_gen1"  : len(batch_report["skipped_gen1"]),
        "skipped_other" : len(batch_report["skipped_other"]),
        "success"       : batch_report["success"],
        "failed"        : batch_report["failed"],
        "skipped_gen1_files"  : batch_report["skipped_gen1"],
        "skipped_other_files" : batch_report["skipped_other"],
    }

    with report_path.open("w", encoding="utf-8") as f:
        json.dump(serializable_report, f, indent=2, ensure_ascii=False)

    print(f"\nBatch report saved: {report_path.name}")


# ── Run Summary ──────────────────────────────────────────────────
print_batch_summary(batch_report)
save_batch_report(batch_report, OUTPUT_FOLDER)

BATCH PROCESSING SUMMARY

File Counts:
  Total files found  : 6
  Successfully parsed: 2
  Failed             : 0
  Skipped (Gen1)     : 4
  Skipped (Other)    : 0

Successfully Parsed Files:
----------------------------------------------------------------------

  BANK (1 files):
     HDFC Bank Limited                             | Q: Q3   | FY: FY2026  | Facts: 134   | Validation: PASSED

  COMMERCIAL (1 files):
     RELIANCE INDUSTRIES LIMITED                   | Q: Q4   | FY: FY2025  | Facts: 302   | Validation: PASSED

Skipped Gen1 Files (to be handled separately):
----------------------------------------------------------------------
    GI_117338_1350247_17012025074725.xml
    INDAS_118123_1365717_30012025030002.xml
    LI_117242_1346294_15012025050606.xml
    NBFC_INDAS_118212_1366575_30012025073535.xml

Overall Statistics (Parsed Files):
----------------------------------------------------------------------
  Total facts extracted      : 436
  Total periods extracted    : 85
 